# 🌿 Resilience Guardian — Custom Crop Disease Model Training
**Developer:** Gulilat Kasiye Worku
**GitHub:** github.com/guleaada

This notebook trains MobileNetV2 models for each Ethiopian crop.
Output: TensorFlow.js models ready for offline PWA deployment.

In [ ]:
# Install dependencies
!pip install tensorflowjs tf2onnx -q
import tensorflow as tf
import tensorflowjs as tfjs
import pathlib, numpy as np
print('TF version:', tf.__version__)

In [ ]:
# ── CONFIG ──────────────────────────────────────────────
CROP = 'enset'  # Change to: teff, wheat, maize, coffee, potato, barley, sorghum
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30

# Disease classes per crop (must match CROP_CLASSES in index.html)
CROP_CLASSES = {
  'enset':   ['Healthy', 'Bacterial_Wilt', 'Root_Mealybug', 'Early_Xcm'],
  'teff':    ['Healthy', 'Leaf_Rust', 'Head_Smudge', 'Shoot_Fly', 'Blister_Blight'],
  'wheat':   ['Healthy', 'Stripe_Rust', 'Stem_Rust', 'Fusarium_Scab', 'Septoria', 'Loose_Smut'],
  'maize':   ['Healthy', 'Northern_Leaf_Blight', 'Fall_Armyworm', 'Common_Rust', 'Streak_Virus', 'Aflatoxin'],
  'coffee':  ['Healthy', 'Berry_Disease', 'Wilt_CWD', 'Leaf_Rust', 'Berry_Borer'],
  'potato':  ['Healthy', 'Late_Blight', 'Bacterial_Wilt', 'Early_Blight', 'Virus_Y'],
  'barley':  ['Healthy', 'Scald', 'Net_Blotch', 'Covered_Smut', 'Stem_Rust'],
  'sorghum': ['Healthy', 'Anthracnose', 'Grain_Mold', 'Head_Smut', 'Shoot_Fly', 'Striga'],
}
NUM_CLASSES = len(CROP_CLASSES[CROP])
print(f'Training: {CROP} | Classes: {CROP_CLASSES[CROP]}')

In [ ]:
# ── DATA LOADING ─────────────────────────────────────────
# Dataset structure: /data/enset/Healthy/*.jpg
#                    /data/enset/Bacterial_Wilt/*.jpg  etc.

# Option 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
data_dir = pathlib.Path(f'/content/drive/MyDrive/RG_Dataset/{CROP}')

# Option 2: Upload zip file
# from google.colab import files
# files.upload()  # Upload your dataset zip
# !unzip dataset.zip -d /content/data

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='training', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=CROP_CLASSES[CROP]
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='validation', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=CROP_CLASSES[CROP]
)
print('Train batches:', len(train_ds), '| Val batches:', len(val_ds))

In [ ]:
# ── DATA AUGMENTATION ────────────────────────────────────
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),  # Important for field photos
    tf.keras.layers.RandomContrast(0.2),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x,y: (augmentation(x, training=True), y)).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
# ── MODEL: MobileNetV2 Transfer Learning ─────────────────
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False, weights='imagenet'
)
base_model.trainable = False  # Freeze base first

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── PHASE 1: Train top layers ─────────────────────────────
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
]
history1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)
print('Phase 1 complete. Val accuracy:', max(history1.history['val_accuracy']))

In [ ]:
# ── PHASE 2: Fine-tune last 30 layers ───────────────────
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # Lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
print('Phase 2 complete. Val accuracy:', max(history2.history['val_accuracy']))

In [ ]:
# ── EXPORT TO TENSORFLOW.JS ──────────────────────────────
import os
output_dir = f'/content/models/{CROP}'
os.makedirs(output_dir, exist_ok=True)

# Save Keras model first
model.save(f'/content/{CROP}_model.h5')

# Convert to TF.js
!tensorflowjs_converter \
    --input_format=keras \
    --output_format=tfjs_graph_model \
    --quantize_uint8 \
    /content/{CROP}_model.h5 \
    /content/models/{CROP}/

print(f'✅ TF.js model saved to {output_dir}')
!ls -lh /content/models/{CROP}/

In [ ]:
# ── SAVE CLASS NAMES (must match CROP_CLASSES in index.html) ──
import json
meta = {
    'crop': CROP,
    'classes': CROP_CLASSES[CROP],
    'version': '2.1',
    'img_size': IMG_SIZE,
    'val_accuracy': round(max(history2.history['val_accuracy']), 4)
}
with open(f'/content/models/{CROP}/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Metadata:', json.dumps(meta, indent=2))

In [ ]:
# ── DOWNLOAD MODEL ───────────────────────────────────────
!zip -r /content/{CROP}_tfjs_model.zip /content/models/{CROP}/
from google.colab import files
files.download(f'/content/{CROP}_tfjs_model.zip')
print('📥 Download started!')
print('Next step: unzip to ResilienceGuardian/public/models/' + CROP + '/')
print('Then: git add public/models && git push')

## 📦 Deployment Instructions

After training:
1. Download the zip file
2. Extract to `ResilienceGuardian/public/models/enset/`
3. Push to GitHub:
```bash
git add public/models/
git commit -m 'Add TF.js model: enset v2.1 (XX% accuracy)'
git push origin main
```
4. The app auto-loads the local model — no code changes needed!

## 📊 Dataset Sources
- Enset: Collect from EIAR Areka / Bonga University
- Teff: EIAR Debre Zeit ARC photo archives  
- Wheat: CIMMYT Ethiopia / Holeta ARC
- Use farmer-submitted photos from the app feedback system!

**Target accuracy per crop:**
- Enset (Bacterial Wilt): 97%+ (Fekadu et al. 2021 achieved 98.87%)
- Teff (Rust + Smudge): 90%+
- All other crops: 85%+